# CCNY Fall 2021 Academic Calendar Scraper

## Stage 0 — Project setup

We will build this notebook one stage at a time, committing each meaningful change.

Source: https://www.ccny.cuny.edu/registrar/fall

**Goal:** a pandas DataFrame with a Python `datetime.date` index named `date` and exactly two columns: `dow` (weekday name) and `text` (calendar explanation).


## Tools and responsibilities

- **requests:** download the HTML.
- **Beautiful Soup:** locate and extract calendar table rows.
- **pandas:** organize records and expand date ranges.
- **datetime:** parse dates and verify Python date objects.
- **Jupyter:** run Python cells and preserve explanations and outputs.
- **Git / GitHub:** record and share successive versions.


## Setup on your computer

From a Terminal opened in this repository, run:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
jupyter notebook
```

Open `fall_2021_calendar.ipynb`, choose the environment's Python kernel, and run cells with **Shift + Enter**.


In [1]:
from datetime import date

example_date = date(2021, 8, 25)
print(example_date)
print(type(example_date))
print(example_date.strftime("%A"))


2021-08-25
<class 'datetime.date'>
Wednesday


## Understand the first cell

`date(2021, 8, 25)` constructs a date from a year, month, and day. It is not a string. `type()` lets us verify its Python type. `strftime("%A")` gives its weekday name.

Expected output:

```text
2021-08-25
<class 'datetime.date'>
Wednesday
```


## Next stages

1. Import the scraping libraries and download the page.
2. Inspect HTML and identify the calendar table.
3. Extract its date, weekday, and explanation strings.
4. Parse dates and expand ranges inclusively.
5. Build and validate the final DataFrame.

**Date policy:** expand each range into one row per day; preserve separate events on overlapping dates; use 2021 when no year is written and preserve explicit years such as January 1, 2022. Document this interpretation because the assignment does not specify its range policy.


In [6]:
from pathlib import Path
from datetime import date, datetime
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://www.ccny.cuny.edu/registrar/fall"
snapshot_path = Path("data/fall_2021_calendar.html")

try:
    response = requests.get(URL, timeout=30)
    response.raise_for_status()

    html = response.content
    source_used = "Live CCNY webpage"

except requests.RequestException as error:
    print("Live download failed:", error)

    html = snapshot_path.read_bytes()
    source_used = "Saved HTML snapshot"

soup = BeautifulSoup(html, "html.parser")
heading = soup.find("h1")
tables = soup.find_all("table")

if heading is None or "Fall 2021 Academic Calendar" not in heading.get_text(" ", strip=True):
    raise ValueError("The HTML does not contain the expected calendar heading.")

print("Source:", source_used)
print("Page heading:", heading.get_text(" ", strip=True))
print("Tables found:", len(tables))

Live download failed: 403 Client Error: Forbidden for url: https://www.ccny.cuny.edu/registrar/fall
Source: Saved HTML snapshot
Page heading: Fall 2021 Academic Calendar
Tables found: 1


In [3]:
calendar_table = tables[0]
rows = calendar_table.find_all("tr")

print("Rows including the header:", len(rows))

for row in rows[:3]:
    cells = row.find_all(["th", "td"])
    print([cell.get_text(" ", strip=True) for cell in cells])

Rows including the header: 37
['DATES', 'DAYS', '']
['August 01', 'Sunday', 'Application for degree for January and February 2022 begins']
['August 18', 'Wednesday', 'Last day to apply for Study Abroad']


In [4]:
import pandas as pd

records = []

for row in rows[1:]:
    cells = row.find_all(["th", "td"])

    values = [
        " ".join(cell.get_text(" ", strip=True).split())
        for cell in cells
    ]

    if len(values) != 3:
        raise ValueError(f"Expected three cells, found: {values}")

    records.append({
        "date_label": values[0],
        "dow_source": values[1],
        "text": values[2],
    })

raw_df = pd.DataFrame(records)

print("Calendar entries:", len(raw_df))
display(raw_df.head())

Calendar entries: 36


,date_label,dow_source,text
0,August 01,Sunday,Application for degree for January and Februar...
1,August 18,Wednesday,Last day to apply for Study Abroad
2,August 24,Tuesday,Last day of Registration; Last day to file ePe...
3,August 25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
4,August 25 - 31,Wednesday - Tuesday,Change of program period; late fees apply


In [7]:
def expand_dates(label):
    label = " ".join(label.split())
    label = label.replace("–", "-").replace("—", "-")

    if "-" in label:
        start_text, end_day_text = label.split("-", 1)

        start = datetime.strptime(
            f"{start_text.strip()} 2021",
            "%B %d %Y"
        ).date()

        end = start.replace(day=int(end_day_text.strip()))

    elif "," in label:
        start = datetime.strptime(label, "%B %d, %Y").date()
        end = start

    else:
        start = datetime.strptime(
            f"{label} 2021",
            "%B %d %Y"
        ).date()

        end = start

    return list(pd.date_range(start=start, end=end, freq="D").date)

In [8]:
print(expand_dates("August 01"))
print(expand_dates("August 25 - 31"))
print(expand_dates("January 1, 2022"))

[datetime.date(2021, 8, 1)]
[datetime.date(2021, 8, 25), datetime.date(2021, 8, 26), datetime.date(2021, 8, 27), datetime.date(2021, 8, 28), datetime.date(2021, 8, 29), datetime.date(2021, 8, 30), datetime.date(2021, 8, 31)]
[datetime.date(2022, 1, 1)]


In [9]:
expanded_records = []

for record in records:
    dates = expand_dates(record["date_label"])

    source_days = (
        record["dow_source"]
        .replace("–", "-")
        .replace("—", "-")
        .split("-")
    )
    source_days = [day.strip() for day in source_days]

    assert dates[0].strftime("%A") == source_days[0], record["date_label"]
    assert dates[-1].strftime("%A") == source_days[-1], record["date_label"]

    for current_date in dates:
        expanded_records.append({
            "date": current_date,
            "dow": current_date.strftime("%A"),
            "text": record["text"],
        })

df = pd.DataFrame(expanded_records)

df = df.set_index("date").sort_index(kind="stable")
df = df[["dow", "text"]]

display(df.head(10))

,dow,text
date,,
2021-08-01,Sunday,Application for degree for January and Februar...
2021-08-18,Wednesday,Last day to apply for Study Abroad
2021-08-24,Tuesday,Last day of Registration; Last day to file ePe...
2021-08-25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
2021-08-25,Wednesday,Change of program period; late fees apply
2021-08-26,Thursday,Change of program period; late fees apply
2021-08-26,Thursday,Last day for Independent Study
2021-08-27,Friday,Change of program period; late fees apply
2021-08-28,Saturday,Change of program period; late fees apply


In [10]:
display(df.loc[date(2021, 8, 25)])


,dow,text
date,,
2021-08-25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
2021-08-25,Wednesday,Change of program period; late fees apply
